In [1]:
from dataset import Dataset
from model import Retriever, Augmenter, Generator, RetrievalEvaluator
from evaluate import Evaluator
import os

import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

In [2]:
note_prompts = {
    "easy": "Important Note: Your output will strictly be Yes or No with no other words or punctuation marks.",
    "medium": "Important Note: Your output must be strictly, with no extra words, separated by comma, a list of nutrients with high or low before the nutrients among these options: carb, protein, sugar, sodium, cholesterol, \
        saturated_fat, calorie. For example, the output is: high_carb, low_protein, high_sugar.\
        You should only include the nutrient tags that connect the food with the user.",
    "hard": "Important Note: Your output must be a Yes or No followed by strictly a list of nutrients with high or low as prefix among these options: carb, protein, sugar, sodium, cholesterol, \
        saturated fat, calorie. For example, the output is: Yes, because the food is high in carb, low in protein, high in sugar.",
}

method_prompts = {
    "plain": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "KAPPING": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "ToG": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "Zero_CoT": "Let's think step by step",
    "CoT_BaG": "Let's construct a graph with the nodes and edges, then provide the output adhering to the following guideline."
}

## ToG

In [ ]:

# Modify the parameters here to find good prompts.
api_key = os.getenv("OPENAI_API_KEY")
# api_key = os.getenv('LLAMA_API_KEY')
file_path = "../processed_data/NutriGraphQA_benchmark.csv"

task_level = "medium"
question_level = "easy"
is_sample = True
n = 10
# model_name = "llama3.1-70b"
model_name = "gpt-4o-mini"
method = "ToG"

note_prompt = note_prompts.get(task_level)
method_prompt = method_prompts.get(method)

data = Dataset(file_path)
questions, answers, graphs = data.process(question_level=question_level, task_level=task_level, sample=is_sample, n=n)

retriever = Retriever(graphs, model_name=model_name)
retrieved_graphs = retriever.retrieve(method=method, api_key=api_key, questions=questions)

retrieval_evaluator = RetrievalEvaluator(graphs)
retrieval_evaluation_results = retrieval_evaluator.evaluate(retrieved_graphs)
print('Retrieval evaluation results:', retrieval_evaluation_results)

augmenter = Augmenter()
textualized_graphs = augmenter.augment(retrieved_graphs)

generator = Generator(api_key = api_key, 
                      model_name=model_name, note_prompt=note_prompt, method_prompt=method_prompt)
predictions = generator.generate_predictions(questions, textualized_graphs)

evaluator = Evaluator()
final_output_results = evaluator.evaluate(task_level, predictions, answers)
print('Final output evaluation results:', final_output_results)

## CoT BaG

In [ ]:
from utils import generate_paragraph_cot_bag

# api_key = os.getenv("OPENAI_API_KEY")
api_key = os.getenv('LLAMA_API_KEY')

file_path = "../processed_data/NutriGraphQA_benchmark.csv"

task_level = "easy"
question_level = "medium"
is_sample = True
n = 50
# model_name = "gpt-4o-mini"
# model_name = "gpt-3.5-turbo"
model_name = "llama3.1-70b"
method = "CoT_BaG"

note_prompt = note_prompts.get(task_level)
method_prompt = method_prompts.get(method)

data = Dataset(file_path)
questions, answers, graphs = data.process(question_level=question_level, task_level=task_level, sample=is_sample, n=n)

retriever = Retriever(graphs, model_name=model_name)
retrieved_graphs = retriever.retrieve(method=method, api_key=api_key, questions=questions)

retrieval_evaluator = RetrievalEvaluator(graphs)
retrieval_evaluation_results = retrieval_evaluator.evaluate(retrieved_graphs)
print('Retrieval evaluation results:', retrieval_evaluation_results)

augmenter = Augmenter()
textualized_graphs = augmenter.augment(retrieved_graphs)

# Convert to paragraph format for CoT_BaG
if method == "CoT_BaG":
    textualized_graphs = [generate_paragraph_cot_bag(graph) for graph in textualized_graphs]

generator = Generator(api_key=api_key, model_name=model_name, note_prompt=note_prompt, method_prompt=method_prompt)
predictions = generator.generate_predictions(questions, textualized_graphs)

evaluator = Evaluator()
final_output_results = evaluator.evaluate(task_level, predictions, answers)
print('Final output evaluation results:', final_output_results)


## CoT

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
# api_key = os.getenv('LLAMA_API_KEY')

file_path = "../processed_data/NutriGraphQA_benchmark.csv"

task_level = "easy"
question_level = "medium"
is_sample = True
n = 50
model_name = "gpt-4o-mini"
# model_name = "gpt-3.5-turbo"
# model_name = "llama3.1-70b"
method = "Zero_CoT"

note_prompt = note_prompts.get(task_level)
method_prompt = method_prompts.get(method)

data = Dataset(file_path)
questions, answers, graphs = data.process(question_level=question_level, task_level=task_level, sample=is_sample, n=n)

retriever = Retriever(graphs, model_name=model_name)
retrieved_graphs = retriever.retrieve(method=method, api_key=api_key, questions=questions)

retrieval_evaluator = RetrievalEvaluator(graphs)
retrieval_evaluation_results = retrieval_evaluator.evaluate(retrieved_graphs)
print('Retrieval evaluation results:', retrieval_evaluation_results)

augmenter = Augmenter()
textualized_graphs = augmenter.augment(retrieved_graphs)

generator = Generator(api_key=api_key, model_name=model_name, note_prompt=note_prompt, method_prompt=method_prompt)
predictions = generator.generate_predictions(questions, textualized_graphs)

evaluator = Evaluator()
final_output_results = evaluator.evaluate(task_level, predictions, answers)
print('Final output evaluation results:', final_output_results)